## 导入config模块路径

In [1]:
import sys
from pathlib import Path
str(Path.cwd().parent) in sys.path or sys.path.append(str(Path.cwd().parent))
print(sys.path)

['c:\\Users\\Leon\\miniforge3\\envs\\cm312\\python312.zip', 'c:\\Users\\Leon\\miniforge3\\envs\\cm312\\DLLs', 'c:\\Users\\Leon\\miniforge3\\envs\\cm312\\Lib', 'c:\\Users\\Leon\\miniforge3\\envs\\cm312', '', 'c:\\Users\\Leon\\miniforge3\\envs\\cm312\\Lib\\site-packages', 'c:\\Users\\Leon\\miniforge3\\envs\\cm312\\Lib\\site-packages\\win32', 'c:\\Users\\Leon\\miniforge3\\envs\\cm312\\Lib\\site-packages\\win32\\lib', 'c:\\Users\\Leon\\miniforge3\\envs\\cm312\\Lib\\site-packages\\Pythonwin', 'c:\\WorkFlow\\E\\Code\\Python\\llm-abilities-toolkit']


## 使用模块管理api-key

In [ ]:
import sys
from pathlib import Path
str(Path.cwd().parent) in sys.path or sys.path.append(str(Path.cwd().parent))
from openai import OpenAI
from config import config

# 使用 ** 解包创建客户端
client = OpenAI(**config.get_modelscope_config())
# client = OpenAI(**config.get_nvidia_config())

# --- 临时覆盖：如需手动指定端点/密钥，取消注释并填写 ---
# client.base_url = "https://token-plan-cn.xiaomimimo.com/v1"
# client.api_key = ""  # ⚠ 不要将密钥提交到版本控制

response = client.chat.completions.create(
    model='deepseek-ai/DeepSeek-V4-Pro', # ModelScope Model-Id, required
    messages=[
        {
            'role': 'user',
            'content': '你好'
        }
    ],
    stream=False
)

print(response.choices[0].message.reasoning_content)
print('\n\n === Final Answer ===\n')
print(response.choices[0].message.content)

好的，用户发来一句简单的问候“你好”。这是一个非常常见的开场白，没有具体问题或指令。我需要判断如何回应最合适。

用户可能只是打个招呼，或者想测试我是否在线。深层需求可能是希望开启一段对话，但尚未明确方向。我应该用友好、开放的方式回应，表达欢迎并提供帮助的意愿，同时引导用户说明具体需求。

想到了用热情的语气回复，加上表情符号传递友好，然后直接表明身份和可用性，最后用一个开放性问题结束，鼓励用户提出具体请求。这样既回应了问候，又为后续互动铺平了道路。


 === Final Answer ===

你好呀！👋 很高兴见到你！

我是DeepSeek，由深度求索公司创造的AI助手。无论你有什么问题、需要什么帮助，或者只是想聊聊天，我都很乐意陪你！

今天有什么我可以帮你的吗？可以是学习、工作、生活中的问题，也可以是创作、翻译、编程等等，尽管说吧！😊


## 接口支持模型

In [ ]:
import sys
from pathlib import Path
str(Path.cwd().parent) in sys.path or sys.path.append(str(Path.cwd().parent))
from openai import OpenAI
from config import config

client = OpenAI(**config.get_modelscope_config())

# --- 临时覆盖：如需手动指定端点/密钥，取消注释并填写 ---
# client.base_url = "https://token-plan-cn.xiaomimimo.com/v1"
# client.api_key = ""  # ⚠ 不要将密钥提交到版本控制

models = client.models.list()

# 收集所有模型ID
model_ids = [model.id for model in models]

# 排序
model_ids.sort()

print(f"端点：{client.base_url}")
print(f"支持的模型列表（共 {len(model_ids)} 个）：")
print("-" * 50)
for model_id in model_ids:
    print(model_id)

端点：https://token-plan-cn.xiaomimimo.com/v1/
支持的模型列表（共 9 个）：
--------------------------------------------------
mimo-v2-omni
mimo-v2-pro
mimo-v2-tts
mimo-v2.5
mimo-v2.5-asr
mimo-v2.5-pro
mimo-v2.5-tts
mimo-v2.5-tts-voiceclone
mimo-v2.5-tts-voicedesign


## models.dev 模型信息查询

In [ ]:
import requests

MODELS_URL = "https://models.dev/models.json"
API_URL = "https://models.dev/api.json"


def get_model_info(model_id: str) -> dict:
    """从 models.dev 查询模型信息
    支持两种格式：
        - "provider/model" 精确查询（大小写不敏感）
        - "model" 在官方库中模糊匹配（大小写不敏感）
    """
    models_data = requests.get(MODELS_URL).json()
    api_data = requests.get(API_URL).json()

    model_id_lower = model_id.lower()

    # 精确匹配（大小写不敏感）
    if "/" in model_id_lower:
        candidates = [k for k in models_data if k.lower() == model_id_lower]
    else:
        # 在 models.json 的 key 中找官方映射（精确匹配 model 名，大小写不敏感）
        candidates = [k for k in models_data if k.split("/")[-1].lower() == model_id_lower]
        if not candidates:
            # 退化为模糊匹配
            candidates = [k for k in models_data if model_id_lower in k.split("/")[-1].lower()]

    canonical_key = candidates[0] if candidates else None

    if not canonical_key or canonical_key not in models_data:
        return None

    result = dict(models_data[canonical_key])

    # 从 api.json 补充价格信息
    provider_id = canonical_key.split("/")[0]
    model_name = canonical_key.split("/", 1)[1]
    provider_info = api_data.get(provider_id, {})
    provider_model = provider_info.get("models", {}).get(model_name, {})
    if "cost" in provider_model:
        result["cost"] = provider_model["cost"]
    result["provider_id"] = provider_id
    result["provider_name"] = provider_info.get("name", "")

    return result


if __name__ == "__main__":
    model_id = "miniMax-m2"
    result = get_model_info(model_id)
    if result:
        print(f"模型: {result.get("id", model_id)}")
        print(f"名称: {result.get("name", "N/A")}")
        print(f"Provider: {result.get("provider_name", "N/A")}")
        print(f"Family: {result.get("family", "N/A")}")
        print(f"知识截止: {result.get("knowledge", "N/A")}")
        print(f"发布日期: {result.get("release_date", "N/A")}")
        print(f"最近更新: {result.get("last_updated", "N/A")}")
        print(f"开源权重: {result.get("open_weights", "N/A")}")
        print("---")
        print(f"支持附件: {result.get("attachment", "N/A")}")
        print(f"支持 Temperature: {result.get("temperature", "N/A")}")
        print(f"支持 Structured Output: {result.get("structured_output", "N/A")}")
        print(f"支持 Tool Call: {result.get("tool_call", "N/A")}")
        print(f"支持 Reasoning: {result.get("reasoning", "N/A")}")
        print("---")
        print(f"输入价格: ${result.get("cost", {}).get("input", "N/A")}/M tokens")
        print(f"输出价格: ${result.get("cost", {}).get("output", "N/A")}/M tokens")
        print(f"缓存读取价格: ${result.get("cost", {}).get("cache_read", "N/A")}/M tokens")
        print("---")
        print(f"上下文窗口: {result.get("limit", {}).get("context", "N/A")} tokens")
        print(f"输出上限: {result.get("limit", {}).get("output", "N/A")} tokens")
        print(f"输入模态: {result.get("modalities", {}).get("input", "N/A")}")
        print(f"输出模态: {result.get("modalities", {}).get("output", "N/A")}")
    else:
        print(f"未找到模型: {model_id}")


模型: minimax/MiniMax-M2
名称: MiniMax-M2
Provider: MiniMax (minimax.io)
Family: minimax
知识截止: N/A
发布日期: 2025-10-27
最近更新: 2025-10-27
开源权重: True
---
支持附件: False
支持 Temperature: True
支持 Structured Output: N/A
支持 Tool Call: True
支持 Reasoning: True
---
输入价格: $0.3/M tokens
输出价格: $1.2/M tokens
缓存读取价格: $N/A/M tokens
---
上下文窗口: 196608 tokens
输出上限: 128000 tokens
输入模态: ['text']
输出模态: ['text']


## 基础调用

### 文本接口三代对比

| 代际 | API 名称              | 接口路径                         | 介绍                          |
| ---- | --------------------- | -------------------------------- | ----------------------------- |
| 第一代 | Completions API       | `/v1/completions`                | 纯文本补全，已废弃            |
| 第二代 | Chat Completions API  | `/v1/chat/completions`           | 对话格式，当前行业标准        |
| 第三代 | Responses API         | `/v1/responses`                  | 2025年发布，面向 Agent 场景   |

### Chat Completions 与 Responses 核心区别

| 特性 | Chat Completions | Responses API |
|------|------------------|---------------|
| **状态管理** | 无状态，需自行维护 | 有状态，服务端存储 |
| **内置工具** | 无 | web_search, file_search, code_interpreter |
| **单次多轮调用** | 不支持 | 支持 |
| **推理痕迹** | 显式推理 | 可获取内部推理 |
| **接口复杂度** | 简单直观 | 稍复杂但更强大 |
| **社区兼容性** | 广泛兼容（行业标准） | OpenAI 专属 |
| **性能** | 更快 | 稍慢（2x-9x 延迟） |
| **适合场景** | 简单对话、兼容性要求高 | Agent、复杂工具链 |


### 直接回复 

In [ ]:
ChatCompletion
├── id: 'chatcmpl-102……'  # 唯一标识符，每次请求生成一个，用于追踪和调试
│
├── object: 'chat.completion'  # 对象类型，固定值（流式时为 'chat.completion.chunk'）
│
├── created: 1782798385  # 服务器响应的 Unix 时间戳（秒）
│
├── model: 'deepseek-ai/DeepSeek-V4-Pro'  # 实际使用的模型 ID
│
├── choices: [Choice]  # 回复选项列表（n>1 时会有多个）
│   └── [0] Choice
│       ├── index: 0  # 当前选项的序号（从 0 开始）
│       │
│       ├── finish_reason: 'stop'
│       │   # 停止原因：
│       │   #   'stop'           → 模型正常结束回复
│       │   #   'length'         → 达到 max_tokens 上限被截断
│       │   #   'tool_calls'     → 模型请求调用工具
│       │   #   'content_filter' → 被内容安全过滤拦截
│       │
│       ├── message: ChatCompletionMessage
│       │   ├── role: 'assistant'  # 消息角色，固定为 'assistant'
│       │   ├── content: '你好呀！👋 ...'  # 最终回复文本（用户看到的内容）
│       │   ├── reasoning_content: '嗯，用户只发了...'  # 推理链/思维过程（推理模型特有）
│       │   ├── refusal: None  # 拒绝内容（模型拒绝回答时填充）
│       │   ├── annotations: None  # 注释信息（引用来源等，大部分模型未启用）
│       │   ├── audio: None  # 音频数据（启用 audio 输出时填充）
│       │   ├── tool_calls: None  # 工具调用列表（Function Calling 时非空）
│       │   ├── function_call: None  # 已废弃，现被 tool_calls 取代
│       │   └── function_calls: None  # 非标准扩展字段
│       │
│       ├── logprobs: None  # token 的对数概率（仅 logprobs=True 时返回）
│       │
│       └── delta:  # 流式输出专用的增量对象
│           ├── role: None  # 流式首块为 'assistant'，后续 None
│           ├── content: ''  # 本次增量文本
│           ├── tool_calls: None  # 增量工具调用
│           ├── function_calls: None  # 增量函数调用
│           └── reasoning_content: ''  # 增量推理内容
│
├── service_tier: None  # 服务等级（standard / priority / batch 等）
│
├── system_fingerprint: ''  # 服务端系统指纹，检测后端配置变更
│
└── usage: CompletionUsage  # ⭐ Token 用量统计，关系到计费
    ├── prompt_tokens: 5  # 输入 token 数
    ├── completion_tokens: 145  # 输出 token 数
    ├── total_tokens: 150  # 总消耗 = prompt + completion
    ├── completion_tokens_details: None
    │   # 输出详情：
    │   #   reasoning_tokens             → 推理消耗的 token
    │   #   accepted_prediction_tokens   → 接受的预测 token
    │   #   rejected_prediction_tokens   → 拒绝的预测 token
    │
    └── prompt_tokens_details: None
        # 输入详情：
        #   cached_tokens → 命中缓存的 token 数（可节省费用）

In [2]:
import sys
from pathlib import Path
str(Path.cwd().parent) in sys.path or sys.path.append(str(Path.cwd().parent))
from openai import OpenAI
from config import config

client = OpenAI(**config.get_modelscope_config())    # 使用 ** 解包创建客户端

response = client.chat.completions.create(
    model='deepseek-ai/DeepSeek-V4-Pro',
    messages=[
        {
            'role': 'user',
            'content': '你好'
        }
    ],
    stream=False
)
print(response)
print('\n\n === Reasoning ===\n')

print(response.choices[0].message.reasoning_content)
print('\n\n === Final Answer ===\n')
print(response.choices[0].message.content)

ChatCompletion(id='chatcmpl-1023a511-80ad-91b6-b0fb-89b8ea7cdc79', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='你好呀！👋 很高兴见到你！\n\n我是DeepSeek，一个由深度求索公司创造的AI助手。无论你想聊什么、问什么，或者需要我帮忙做什么，我都非常乐意陪伴你、帮助你！\n\n今天有什么我可以为你效劳的吗？😊', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, function_calls=None, reasoning_content='嗯，用户只发了“你好”两个字，这是一个非常简单的开场问候。\n\n我需要给出一个友好、热情的回应，并尽快引导对话进入实质内容。可以用打招呼+自我介绍+开放询问的组合。\n\n想到了用“你好呀”这样亲切的问候，加上表情符号增加亲和力。然后简单介绍我是谁，能做什么，最后用一个开放性问题把话题抛回给用户，这样对话就能自然延续下去了。'), delta={'role': None, 'content': '', 'tool_calls': None, 'function_calls': None, 'reasoning_content': ''})], created=1782798385, model='deepseek-ai/DeepSeek-V4-Pro', object='chat.completion', service_tier=None, system_fingerprint='', usage=CompletionUsage(completion_tokens=145, prompt_tokens=5, total_tokens=150, completion_tokens_details=None, prompt_tokens_details=None))
嗯，用户只发了“你好”两个字，这是一个非常简单的开场问候。

我需要给出一个友好、热情的回应

### 流式输出

In [ ]:
完整响应（stream=False）                流式 chunk（stream=True）
ChatCompletion                         ChatCompletionChunk
├── id: 'chatcmpl-xxx'                ├── id: 'chatcmpl-xxx' # 所有 chunk 共享同一个 id
├── object: 'chat.completion'          ├── object: 'chat.completion.chunk'
├── created: 1782798385                ├── created: 1782801089
├── model: 'deepseek-ai/DeepSeek-...' ├── model: 'deepseek-ai/DeepSeek-...'
│                                      │
├── choices:                           ├── choices:
│   └── [0]                            │   └── [0]
│       ├── index: 0                   │       ├── index: 0
│       ├── finish_reason: 'stop'      │       ├── finish_reason: null # 仅最后一个chunk有值
│       │                              │       │
│       └── message: ChatCompletion... │       ├── delta: ChoiceDelta # ⭐增量数据在delta里
│           ├── role: 'assistant'      │       │   ├── role: 'assistant' # 仅首个chunk有值
│           ├── content: '完整回复...'  │       │   ├── content: '你'   # 1~2 个 token 的增量
│           ├── reasoning_content: ... │       │   ├── reasoning_content: '嗯'  # 推理模型的思考增量
│           ├── refusal: None          │       │   ├── refusal: None
│           ├── annotations: None      │       │   ├── function_call: None    # 已废弃
│           ├── audio: None            │       │   ├── tool_calls: None # Function Calling时非空
│           ├── tool_calls: None       │       │   └── function_calls: None   # 非标准扩展
│           ├── function_call: None    │       │    # ⚠️ delta 无 annotations/audio
│           └── function_calls: None   │       │
│                                      │       └── message: ChoiceMessage   # ⬅️ 存在但是空占位，无实际用途
│                                      │           ├── role: None
│                                      │           ├── content: ''
│                                      │           ├── tool_calls: None
│                                      │           ├── function_calls: None
│                                      │           └── reasoning_content: ''
│                                      │
├── service_tier: None                 ├── service_tier: None
├── system_fingerprint: ''             ├── system_fingerprint: ''
│                                      │
└── usage: CompletionUsage             └── usage: CompletionUsage          # ⬅️ 不是 None，是全零对象
    ├── prompt_tokens: 5                   ├── prompt_tokens: 0              # 每个 chunk 都是 0
    ├── completion_tokens: 145             ├── completion_tokens: 0
    ├── total_tokens: 150                  ├── total_tokens: 0
    ├── completion_tokens_details: None    ├── completion_tokens_details: None
    └── prompt_tokens_details: None        └── prompt_tokens_details: None
    # 有真实的 token 统计                 # 全零，部分 API 最后一个 chunk 返回真实值

In [9]:
import sys
from pathlib import Path
str(Path.cwd().parent) in sys.path or sys.path.append(str(Path.cwd().parent))
from openai import OpenAI
from config import config

client = OpenAI(**config.get_modelscope_config())    # 使用 ** 解包创建客户端

response = client.chat.completions.create(  # 流式请求，返回可迭代的 chunk 对象
    model='deepseek-ai/DeepSeek-V4-Pro', # ModelScope Model-Id, required
    messages=[
        {
            'role': 'user',
            'content': '你好'
        }
    ],
    stream=True  # 启用流式输出，数据分块到达
)
print(response)
print('\n\n === Reasoning ===\n')
done_reasoning = False                     # 标记推理阶段是否结束，用于在推理→回答切换时插入分隔线
for chunk in response:                     # 每次迭代获取一个增量数据块
    if chunk.choices:                      # choices 通常只有一个元素（由 n 参数控制）
        reasoning_chunk = chunk.choices[0].delta.reasoning_content  # 模型的思考过程（仅 DeepSeek 等推理模型有）
        answer_chunk = chunk.choices[0].delta.content               # 最终回答的增量文本
        if reasoning_chunk != '':           # 当前 chunk 是推理内容，直接输出
            print(reasoning_chunk, end='', flush=True)
        elif answer_chunk != '':            # 当前 chunk 是回答内容
            if not done_reasoning:          # 推理刚结束，首次进入回答阶段
                print('\n\n === Final Answer ===\n')
                done_reasoning = True
            print(answer_chunk, end='', flush=True)  # flush=True 确保逐字符实时显示



 === Reasoning ===

嗯，用户只发了“你好”两个字，这是一个非常简单的问候。

我需要判断如何回应。用户可能是在测试连接，或者只是开始一段对话。深层需求可能是想确认我是否在线、能否正常交流，或者只是友好地打个招呼。

考虑到这是对话的起点，我应该用热情、友好的语气回应，并主动介绍自己，表示可以提供帮助，这样能鼓励用户提出后续问题。

我可以这样开始回复。

 === Final Answer ===

你好！很高兴见到你！😊

我是DeepSeek，你的AI助手，可以帮你解答问题、提供建议、聊天或者处理各种任务。有什么我可以帮你的吗？无论是学习、工作还是生活中的问题，都可以随时问我哦！

## 魔搭剩余次数

- 可以用async_openai

In [ ]:
import sys
from pathlib import Path
str(Path.cwd().parent) in sys.path or sys.path.append(str(Path.cwd().parent))
from openai import OpenAI, APITimeoutError
from config import config
from IPython.display import display, HTML

REQUEST_TIMEOUT = 30
MAX_ATTEMPTS = 2  # 首次请求 + 1 次重试

# 使用 ** 解包创建客户端
client = OpenAI(**config.get_modelscope_config(), timeout=REQUEST_TIMEOUT)

models = [
    "Qwen/Qwen3-235B-A22B-Instruct-2507",
    "Qwen/Qwen3-VL-235B-A22B-Instruct",
    "Qwen/Qwen3.5-397B-A17B",
    "Qwen/Qwen3.5-27B",
    "moonshotai/Kimi-K2.5",
    "ZhipuAI/GLM-5.1",
    "MiniMax/MiniMax-M3",
    "deepseek-ai/DeepSeek-V4-Pro",
    "inclusionAI/Ling-2.6-1T",
    "inclusionAI/Ring-2.6-1T"
]

_status = display(HTML(""), display_id="model_rate_limit_status")

def set_status(msg: str):
    _status.update(HTML(
        f"<pre style='margin:0 0 8px 0;font-family:monospace;color:#6cb6ff'>"
        f"状态 │ {msg}"
        f"</pre>"
    ))

set_status("准备开始...")
print(f"{'模型':<40} {'模型剩余/上限':<12} {'账号剩余/上限'}")
print("-" * 73, flush=True)

results = []
ok_count = fail_count = 0
total = len(models)

for idx, model_id in enumerate(models, 1):
    row = None
    last_error = None
    progress = f"[{idx}/{total}]"

    for attempt in range(MAX_ATTEMPTS):
        if attempt == 0:
            set_status(f"测试中 {progress} · {model_id}")
        else:
            set_status(f"重试中 {progress} · {model_id} · 上次超时 {REQUEST_TIMEOUT}s")

        try:
            raw = client.chat.completions.with_raw_response.create(
                model=model_id,
                messages=[{"role": "user", "content": "hi"}],
                max_tokens=1,
                timeout=REQUEST_TIMEOUT,
            )
            h = raw.headers
            row = {
                "model": model_id,
                "model_remaining": h.get("modelscope-ratelimit-model-requests-remaining", "N/A"),
                "model_limit":     h.get("modelscope-ratelimit-model-requests-limit", "N/A"),
                "user_remaining":  h.get("modelscope-ratelimit-requests-remaining", "N/A"),
                "user_limit":      h.get("modelscope-ratelimit-requests-limit", "N/A"),
            }
            break
        except APITimeoutError as e:
            last_error = e
        except Exception as e:
            last_error = e
            break

    if row:
        results.append(row)
        ok_count += 1
        model_col = f"{row['model_remaining']}/{row['model_limit']}"
        user_col  = f"{row['user_remaining']}/{row['user_limit']}"
        print(f"{model_id:<45} {model_col:<15} {user_col}", flush=True)
    else:
        results.append({"model": model_id, "error": str(last_error)})
        fail_count += 1
        print(f"{model_id:<45} 请求失败: {last_error}", flush=True)

set_status(f"测试全部完成 · 共 {total} 个模型 · 成功 {ok_count} · 失败 {fail_count}")

模型                                       模型剩余/上限      账号剩余/上限
-------------------------------------------------------------------------
Qwen/Qwen3-235B-A22B-Instruct-2507            49/50           1998/2000
Qwen/Qwen3-VL-235B-A22B-Instruct              99/100          1997/2000
Qwen/Qwen3.5-397B-A17B                        98/100          1996/2000
Qwen/Qwen3.5-27B                              199/200         1995/2000
moonshotai/Kimi-K2.5                          49/50           1994/2000
ZhipuAI/GLM-5.1                               49/50           1993/2000
MiniMax/MiniMax-M3                            199/200         1992/2000
deepseek-ai/DeepSeek-V4-Pro                   49/50           1991/2000
inclusionAI/Ling-2.6-1T                       199/200         1990/2000
inclusionAI/Ring-2.6-1T                       99/100          1989/2000


## 接口能力测试

In [2]:
import sys
import time
import json
import base64
import unicodedata
import requests
from pathlib import Path
from openai import OpenAI

str(Path.cwd().parent) in sys.path or sys.path.append(str(Path.cwd().parent))
from config import config

MAX_ATTEMPTS = 2  # 首次请求 + 1 次重试
MAX_TOKENS = 500  # 推理模型需要较多 token（reasoning + content）


# ──────────────────────────────────────────────
# 工具函数
# ──────────────────────────────────────────────
TEST_NAMES = {
    1: "基础对话", 2: "System message", 3: "多轮对话",
    4: "流式输出 (streaming)", 5: "JSON Mode", 6: "Structured Outputs",
    7: "Function Calling", 8: "并行工具调用", 9: "Vision 图片(URL)",
    10: "Base64 图片", 11: "temperature", 12: "top_p",
    13: "n 参数", 14: "seed", 15: "logit_bias",
    16: "frequency_penalty", 17: "presence_penalty",
    18: "stop 参数", 19: "logprobs", 20: "Usage 统计",
}


def display_width(s):
    return sum(2 if unicodedata.east_asian_width(c) in ("F", "W") else 1 for c in s)


def pad_right(s, width):
    return s + " " * (width - display_width(s))


def call_api(client, **kwargs):
    """带重试的 API 调用，返回 (response, None) 或 (None, error_str)

    对以下情况自动重试:
    - choices 为 null（API 返回 200 但无内容，可能是限流）
    - 网络异常
    """
    for attempt in range(MAX_ATTEMPTS):
        try:
            resp = client.chat.completions.create(**kwargs)
            # choices: null 可能是一次性请求参数过多导致的限流
            if resp and resp.choices is None:
                if attempt < MAX_ATTEMPTS - 1:
                    time.sleep(2)
                    continue
                return resp, "API返回choices:null(可能不支持所请求的参数组合)"
            return resp, None
        except Exception as e:
            if attempt == MAX_ATTEMPTS - 1:
                return None, str(e)
            time.sleep(2)  # 重试前等待
    return None, "unknown"


def get_content(resp):
    """从响应中提取文本，兼容推理模型（reasoning model）。

    推理模型的 content 可能为 null，实际输出在 reasoning 字段。
    优先返回 content，若 content 为空则返回 reasoning 的内容。
    """
    if resp is None or not resp.choices:
        return ""
    msg = resp.choices[0].message
    # 优先取 content
    content = getattr(msg, "content", None) or ""
    if content.strip():
        return content
    # 推理模型：尝试多个字段 (reasoning: mimo/deepseek, reasoning_content: Qwen)
    for field in ("reasoning", "reasoning_content"):
        val = getattr(msg, field, None) or ""
        if val.strip():
            return val
    return ""


def get_finish_reason(resp):
    """获取 finish_reason"""
    if resp is None or not resp.choices:
        return None
    return resp.choices[0].finish_reason


def has_content(resp):
    """检查响应是否有实际 content（不只是 reasoning）"""
    if resp is None or not resp.choices:
        return False
    msg = resp.choices[0].message
    return bool((getattr(msg, "content", None) or "").strip())


def is_vision_refusal(content):
    """检查模型是否以文本形式拒绝处理图片（声称自己无法看图）"""
    lower = (content or "").lower()
    refusal_phrases = [
        "无法查看", "无法看到", "无法识别", "无法处理", "无法分析",
        "不能查看", "不能看到", "不能识别", "不能处理", "不能分析",
        "无法直接查看", "无法直接看到", "无法直接识别",
        "不支持图片", "不支持视觉", "不支持图像",
        "cannot view", "cannot see", "cannot process", "cannot analyze",
        "cannot view images", "unable to view", "unable to see",
        "text-based", "文本的ai", "基于文本的",
    ]
    return any(phrase in lower for phrase in refusal_phrases)


def serialize_response(resp):
    """输出完整原始 response JSON"""
    if resp is None:
        return "(空响应)"
    try:
        return json.dumps(resp.model_dump(), ensure_ascii=False, indent=2)
    except Exception:
        return repr(resp)


# ──────────────────────────────────────────────
# 结果收集: 每次 API 调用产出一个 CallGroup
# ──────────────────────────────────────────────
class CallGroup:
    """一次 API 调用的完整结果"""

    def __init__(self, label: str):
        self.label = label
        # [(test_idx, status, message)]  status: "OK" | "PARTIAL" | "FAIL" | "NOT SUPPORTED"
        self.items: list[tuple[int, str, str]] = []
        self.raw: str | None = None

    def add(self, test_idx: int, status: str, message: str):
        self.items.append((test_idx, status, message))

    def set_raw(self, raw: str):
        self.raw = raw

    @property
    def has_problem(self) -> bool:
        return any(s != "OK" for _, s, _ in self.items)

    @property
    def problem_summary(self) -> str:
        """按类型汇总非OK项: NOT SUPPORTED: xxx；PARTIAL: xxx；FAIL: xxx"""
        buckets: dict[str, list[str]] = {
            "NOT SUPPORTED": [], "PARTIAL": [], "FAIL": []
        }
        for idx, status, _ in self.items:
            if status in buckets:
                buckets[status].append(TEST_NAMES[idx])
        parts = []
        for label, names in buckets.items():
            if names:
                parts.append(f"{label}: " + "、".join(names))
        return "；".join(parts) + "。" if parts else ""


# ──────────────────────────────────────────────
# 输出格式化
# ──────────────────────────────────────────────
def show_summary_table(groups: list[CallGroup]):
    """按序号排序输出汇总表格"""
    all_items = []
    for g in groups:
        for idx, _status, message in g.items:
            all_items.append((idx, message))
    all_items.sort(key=lambda x: x[0])

    print(f"{'序号':>4}  {pad_right('功能', 25)} {'结果'}")
    print("-" * 80)
    for idx, message in all_items:
        name = TEST_NAMES[idx]
        display = message[:32].replace("\n", "\\n")    # 表格显示前30个字符
        print(f"{idx:>4}. {pad_right(name, 25)} {display}")


def show_raw_responses(groups: list[CallGroup]):
    """按调用顺序输出有问题的调用详情及 raw response"""
    problem_groups = [g for g in groups if g.has_problem]
    if not problem_groups:
        return

    print()
    print("=" * 80)
    print("raw response:")
    print("=" * 80)

    for g in problem_groups:
        print(f"\n{g.label}")
        print(f"  {g.problem_summary}")
        if g.raw:
            print("  【raw response】")
            print("  " + "-" * 76)
            for line in g.raw.split("\n"):
                print(f"  {line}")
            print("  " + "-" * 76)


# ──────────────────────────────────────────────
# 测试主逻辑
# ──────────────────────────────────────────────
def main(client, model, msgs_basic, msgs_multi, msgs_stream, msgs_json,
            msgs_struct, tools, tool_img_url, img_url):

    groups: list[CallGroup] = []

    # ==========================================
    # 第1次调用: 基础对话 + System + 多参数
    # ==========================================
    g = CallGroup(
        "第1次调用: 基础对话 + System + temperature + top_p + seed "
        "+ frequency_penalty + presence_penalty + logprobs + usage"
    )
    resp, err = call_api(client,
        model=model, messages=msgs_basic, max_tokens=MAX_TOKENS,
        temperature=0.0, top_p=1.0, seed=42,
        frequency_penalty=0.5, presence_penalty=0.5,
        logprobs=True, top_logprobs=2,
    )
    if resp and resp.choices:
        content = get_content(resp)
        finish = get_finish_reason(resp)
        lp = resp.choices[0].logprobs
        
        # 检查是否有实际内容（对齐 Test 3 的逻辑）
        if not content.strip():
            g.add(1, "FAIL", f"FAIL: 响应内容为空 (finish={finish})")
        else:
            g.add(1, "OK", f"OK: {content.strip()[:80]}")

        sys_ok = content.strip().lower() in ("yes", "yes.", "'yes'", '"yes"', "'yes'.", '"yes".')
        g.add(2, "OK" if sys_ok else "PARTIAL",
                f"{'OK' if sys_ok else 'PARTIAL'}: {content.strip()[:80]}")

        for idx in [11, 12, 14, 16, 17]:
            g.add(idx, "OK", "OK: 参数已接受")

        # 推理模型通常不支持 logprobs，logprobs 为 null 属正常
        if finish == "length":
            g.add(19, "NOT SUPPORTED",
                    "NOT SUPPORTED: 推理模型不支持 logprobs (finish=length, "
                    "reasoning 消耗了 token 预算)")
        elif lp and lp.content:
            g.add(19, "OK", f"OK: 有{len(lp.content)}个token的logprobs")
        else:
            g.add(19, "NOT SUPPORTED", "NOT SUPPORTED: logprobs 返回 null")

        if resp.usage:
            u = resp.usage
            reasoning_tokens = getattr(
                getattr(u, "completion_tokens_details", None) or {},
                "reasoning_tokens", 0) or 0
            g.add(20, "OK",
                    f"OK: prompt={u.prompt_tokens}, "
                    f"completion={u.completion_tokens} "
                    f"(reasoning={reasoning_tokens}), "
                    f"total={u.total_tokens}")
        else:
            g.add(20, "PARTIAL", "PARTIAL: usage 为空")

        g.set_raw(serialize_response(resp))
    else:
        e = err or "API返回空响应"
        for idx in [1, 2, 11, 12, 14, 16, 17, 19, 20]:
            g.add(idx, "FAIL", f"FAIL: {e}")
        g.set_raw(serialize_response(resp) if resp else e)
    groups.append(g)

    # ==========================================
    # 第2次调用: n 参数
    # ==========================================
    g = CallGroup("第2次调用: n 参数")
    resp, err = call_api(client,
        model=model, messages=[{"role": "user", "content": "说一个数字"}],
        max_tokens=MAX_TOKENS, n=2,
    )
    if resp and resp.choices:
        n_count = len(resp.choices)
        content = get_content(resp)
        if n_count > 1:
            g.add(13, "OK", f"OK: 返回{n_count}个回复")
        else:
            finish = get_finish_reason(resp)
            reason = (f", finish={finish}" if finish else "")
            g.add(13, "NOT SUPPORTED",
                    f"NOT SUPPORTED: 仅返回1个回复{reason}（推理模型可能不支持n参数）")
        g.set_raw(serialize_response(resp))
    else:
        e = err or "API返回空响应"
        g.add(13, "NOT SUPPORTED", f"NOT SUPPORTED: {e[:80]}")
        g.set_raw(serialize_response(resp) if resp else e)
    groups.append(g)

    # ==========================================
    # 第3次调用: logit_bias
    # ==========================================
    g = CallGroup("第3次调用: logit_bias")
    resp, err = call_api(client,
        model=model, messages=[{"role": "user", "content": "说yes"}],
        max_tokens=MAX_TOKENS, logit_bias={100: 5},
    )
    if resp and resp.choices:
        g.add(15, "OK", "OK: 参数已接受")
        g.set_raw(serialize_response(resp))
    else:
        e = err or "API返回空响应"
        g.add(15, "NOT SUPPORTED", f"NOT SUPPORTED: {e[:80]}")
        g.set_raw(serialize_response(resp) if resp else e)
    groups.append(g)

    # ==========================================
    # 第4次调用: 多轮对话 + stop
    # ==========================================
    g = CallGroup("第4次调用: 多轮对话 + stop")
    resp, err = call_api(client,
        model=model, messages=msgs_multi, max_tokens=MAX_TOKENS, stop=["。"],
    )
    if resp and resp.choices:
        content = get_content(resp).strip()
        finish = get_finish_reason(resp)
        has_name = "小明" in content
        # 推理模型可能只输出 reasoning 而不输出 content
        if not has_content(resp) and content:
            g.add(3, "PARTIAL",
                    f"PARTIAL: 模型仅输出 reasoning，无实际 content: "
                    f"{content[:60]}")
        else:
            g.add(3, "OK" if has_name else "FAIL",
                    f"{'OK' if has_name else 'FAIL'}: {content[:80]}")
        g.add(18, "OK", f"OK: finish={finish}, content={content[:80]!r}")
        g.set_raw(serialize_response(resp))
    else:
        e = err or "API返回空响应"
        g.add(3, "FAIL", f"FAIL: {e}")
        g.add(18, "FAIL", f"FAIL: {e}")
        g.set_raw(serialize_response(resp) if resp else e)
    groups.append(g)

    # ==========================================
    # 第5次调用: 流式输出
    # ==========================================
    g = CallGroup("第5次调用: 流式输出 (streaming)")
    stream_ok = False
    last_err = ""
    for _ in range(MAX_ATTEMPTS):
        try:
            stream = client.chat.completions.create(
                model=model, messages=msgs_stream, max_tokens=MAX_TOKENS,
                stream=True,
            )
            text_parts = []
            reasoning_parts = []
            for chunk in stream:
                if not chunk.choices:
                    continue
                delta = chunk.choices[0].delta
                # 收集 content
                if delta.content:
                    text_parts.append(delta.content)
                # 推理模型：收 reasoning / reasoning_content 字段
                for field in ("reasoning", "reasoning_content"):
                    val = getattr(delta, field, None)
                    if val:
                        reasoning_parts.append(val)
            # 优先用 content，没有则用 reasoning
            full_text = "".join(text_parts) or "".join(reasoning_parts)
            if full_text.strip():
                g.add(4, "OK", f"OK: {full_text.strip()[:80]}")
                g.set_raw(f"[stream content] {full_text.strip()[:200]}")
                stream_ok = True
            else:
                g.add(4, "PARTIAL", "PARTIAL: 流式返回为空")
                g.set_raw("[stream] empty response")
                stream_ok = True
            break
        except Exception as e:
            last_err = str(e)
    if not stream_ok:
        g.add(4, "FAIL", f"FAIL: {last_err}")
        g.set_raw(last_err)
    groups.append(g)

    # ==========================================
    # 第6次调用: JSON Mode
    # ==========================================
    g = CallGroup("第6次调用: JSON Mode")
    resp, err = call_api(client,
        model=model, messages=msgs_json, max_tokens=MAX_TOKENS,
        response_format={"type": "json_object"},
    )
    if resp and resp.choices:
        raw = get_content(resp).strip()
        if not raw:
            g.add(5, "PARTIAL", "PARTIAL: 返回为空（推理模型 content 为 null）")
        else:
            try:
                parsed = json.loads(raw)
                if isinstance(parsed, dict) and parsed.get("value") == 42:
                    extra = [k for k in parsed if k != "value"]
                    if extra:
                        g.add(5, "PARTIAL",
                                f"PARTIAL: 包含额外字段 {extra}: {raw[:80]}")
                    else:
                        g.add(5, "OK", f"OK: {raw[:80]}")
                else:
                    g.add(5, "PARTIAL",
                            f"PARTIAL: 内容不符合要求 {{\"value\": 42}}: {raw[:80]}")
            except json.JSONDecodeError as e:
                g.add(5, "PARTIAL", f"PARTIAL: JSON解析失败: {e}")
        g.set_raw(serialize_response(resp))
    else:
        e = err or "API返回空响应"
        g.add(5, "FAIL", f"FAIL: {e}")
        g.set_raw(serialize_response(resp) if resp else e)
    groups.append(g)

    # ==========================================
    # 第7次调用: Structured Outputs
    # ==========================================
    g = CallGroup("第7次调用: Structured Outputs")
    resp, err = call_api(client,
        model=model, messages=msgs_struct, max_tokens=MAX_TOKENS,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "weather",
                "strict": True,
                "schema": {
                    "type": "object",
                    "properties": {
                        "city": {"type": "string"},
                        "weather": {"type": "string"},
                    },
                    "required": ["city", "weather"],
                    "additionalProperties": False,
                },
            },
        },
    )
    if resp and resp.choices:
        raw = get_content(resp).strip()
        if not raw:
            g.add(6, "PARTIAL", "PARTIAL: 返回为空（推理模型 content 为 null）")
        else:
            try:
                parsed = json.loads(raw)
                # 检查是否严格遵循了 schema（必须包含 city 和 weather）
                ok = isinstance(parsed, dict) and "city" in parsed and "weather" in parsed
                has_extra = ok and any(
                    k not in ("city", "weather") for k in parsed.keys())
                if ok and not has_extra:
                    g.add(6, "OK", f"OK: {raw[:80]}")
                elif ok and has_extra:
                    g.add(6, "PARTIAL",
                            f"PARTIAL: 包含额外字段（模型未完全遵循 schema）: "
                            f"{raw[:80]}")
                else:
                    g.add(6, "PARTIAL",
                            f"PARTIAL: 未遵循 schema（缺少必要字段）: "
                            f"{raw[:80]}")
            except json.JSONDecodeError as e:
                g.add(6, "PARTIAL", f"PARTIAL: JSON解析失败: {e}")
        g.set_raw(serialize_response(resp))
    else:
        e = err or "API返回空响应"
        g.add(6, "FAIL", f"FAIL: {e}")
        g.set_raw(serialize_response(resp) if resp else e)
    groups.append(g)

    # ==========================================
    # 第8次调用: Function Calling + 并行工具调用
    # ==========================================
    g = CallGroup("第8次调用: Function Calling + 并行工具调用")
    resp, err = call_api(client,
        model=model,
        messages=[{"role": "user", "content": "北京今天天气怎么样，现在几点了"}],
        max_tokens=MAX_TOKENS, tools=tools, tool_choice="auto",
        parallel_tool_calls=True,
    )
    if resp and resp.choices:
        msg = resp.choices[0].message
        if msg.tool_calls:
            calls_count = len(msg.tool_calls)
            args = msg.tool_calls[0].function.arguments
            try:
                parsed_args = json.loads(args)
                if isinstance(parsed_args, dict) and "city" in parsed_args:
                    g.add(7, "OK", f"OK: {args[:80]}")
                elif isinstance(parsed_args, dict):
                    g.add(7, "PARTIAL",
                            f"PARTIAL: 缺少 city 字段: {args[:80]}")
                else:
                    g.add(7, "PARTIAL",
                            f"PARTIAL: 参数格式异常(期望dict): {args[:80]}")
            except json.JSONDecodeError as e:
                g.add(7, "PARTIAL", f"PARTIAL: arguments 非合法JSON: {e}")
            g.add(8, "OK" if calls_count > 1 else "PARTIAL",
                    f"{'OK' if calls_count > 1 else 'PARTIAL'}: {calls_count}个工具调用")
        else:
            g.add(7, "PARTIAL", "PARTIAL: 未触发工具调用")
            g.add(8, "PARTIAL", "PARTIAL: 未触发工具调用")
        g.set_raw(serialize_response(resp))
    else:
        e = err or "API返回空响应"
        g.add(7, "FAIL", f"FAIL: {e}")
        g.add(8, "FAIL", f"FAIL: {e}")
        g.set_raw(serialize_response(resp) if resp else e)
    groups.append(g)

    # ==========================================
    # 第9次调用: Vision 图片(URL)
    # ==========================================
    g = CallGroup("第9次调用: Vision 图片(URL)")
    resp, err = call_api(client,
        model=model, messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": "描述这张图片的内容"},
                {"type": "image_url", "image_url": {"url": tool_img_url}},
            ],
        }], max_tokens=MAX_TOKENS,
    )
    if resp and resp.choices:
        content = get_content(resp)
        if content.strip() and not is_vision_refusal(content):
            g.add(9, "OK", f"OK: {content[:50]}")
        elif content.strip() and is_vision_refusal(content):
            g.add(9, "NOT SUPPORTED",
                    f"NOT SUPPORTED: 模型以文本形式拒绝处理图片: {content[:40]}")
        else:
            g.add(9, "NOT SUPPORTED", "NOT SUPPORTED: 模型不支持图片输入")
        g.set_raw(serialize_response(resp))
    else:
        e = err or "API返回空响应"
        g.add(9, "NOT SUPPORTED", f"NOT SUPPORTED: {e[:80]}")
        g.set_raw(serialize_response(resp) if resp else e)
    groups.append(g)

    # ==========================================
    # 第10次调用: Base64 图片
    # ==========================================
    g = CallGroup("第10次调用: Base64 图片")
    try:
        img_b64 = base64.b64encode(requests.get(img_url, timeout=10).content).decode()
    except Exception as e:
        g.add(10, "FAIL", f"FAIL: 图片下载失败 {e}")
        g.set_raw(str(e))
        img_b64 = None

    if img_b64:
        resp, err = call_api(client,
            model=model, messages=[{
                "role": "user",
                "content": [
                    {"type": "text", "text": "描述这张图片的内容"},
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_b64}"}},
                ],
            }], max_tokens=MAX_TOKENS,
        )
        if resp and resp.choices:
            content = get_content(resp)
            if content.strip() and not is_vision_refusal(content):
                g.add(10, "OK", f"OK: {content[:50]}")
            elif content.strip() and is_vision_refusal(content):
                g.add(10, "NOT SUPPORTED",
                        f"NOT SUPPORTED: 模型以文本形式拒绝处理图片: {content[:40]}")
            else:
                g.add(10, "NOT SUPPORTED", "NOT SUPPORTED: 模型不支持图片输入")
            g.set_raw(serialize_response(resp))
        else:
            e = err or "API返回空响应"
            g.add(10, "NOT SUPPORTED", f"NOT SUPPORTED: {e[:80]}")
            g.set_raw(serialize_response(resp) if resp else e)
    groups.append(g)

    # ==========================================
    # 输出
    # ==========================================
    show_summary_table(groups)
    show_raw_responses(groups)


# ──────────────────────────────────────────────
# 可变参数 — 改这里就行，改完直接运行
# ──────────────────────────────────────────────
if __name__ == "__main__":
    MODEL = "Qwen/Qwen3.5-27B"

    # 客户端配置
    client = OpenAI(**config.get_modelscope_config())
    # --- 临时覆盖：如需手动指定端点/密钥，取消注释并填写 ---
    # client.base_url = "https://opencode.ai/zen/v1"
    # client.api_key = ""  # ⚠ 不要将密钥提交到版本控制

    # 第1次调用: 基础对话 + System + 多参数
    MSGS_BASIC = [
        {"role": "system", "content": "你只能回复'yes'"},
        {"role": "user", "content": "明白吗"},
    ]

    # 第4次调用: 多轮对话 + stop
    MSGS_MULTI = [
        {"role": "user", "content": "我叫小明"},
        {"role": "assistant", "content": "你好小明"},
        {"role": "user", "content": "我叫什么名字，回复完整名字即可"},
    ]

    # 第5次调用: 流式输出
    MSGS_STREAM = [{"role": "user", "content": "数1到3，每个数字一行"}]

    # 第6次调用: JSON Mode
    MSGS_JSON = [
        {"role": "system", "content": "只输出JSON"},
        {"role": "user", "content": '返回 {"value": 42}'},
    ]

    # 第7次调用: Structured Outputs
    MSGS_STRUCT = [{"role": "user", "content": "北京的天气"}]

    # 第8次调用: Function Calling
    TOOLS = [
        {
            "type": "function",
            "function": {
                "name": "get_weather",
                "description": "获取城市天气",
                "parameters": {
                    "type": "object",
                    "properties": {"city": {"type": "string", "description": "城市名"}},
                    "required": ["city"],
                },
            },
        },
        {
            "type": "function",
            "function": {
                "name": "get_time",
                "description": "获取城市当前时间",
                "parameters": {
                    "type": "object",
                    "properties": {"city": {"type": "string", "description": "城市名"}},
                    "required": ["city"],
                },
            },
        },
    ]

    # 第9次调用: Vision
    TOOL_IMG_URL = "https://img-blog.csdnimg.cn/img_convert/60d9dae4741f7df4874da06a33cfe05a.png"
    # 第10次调用: Base64 图片
    IMG_URL = TOOL_IMG_URL

    main(
        client=client,
        model=MODEL,
        msgs_basic=MSGS_BASIC,
        msgs_multi=MSGS_MULTI,
        msgs_stream=MSGS_STREAM,
        msgs_json=MSGS_JSON,
        msgs_struct=MSGS_STRUCT,
        tools=TOOLS,
        tool_img_url=TOOL_IMG_URL,
        img_url=IMG_URL,
    )

  序号  功能                      结果
--------------------------------------------------------------------------------
   1. 基础对话                  OK: yes
   2. System message            OK: yes
   3. 多轮对话                  OK: 小明
   4. 流式输出 (streaming)      OK: 1\n2\n3
   5. JSON Mode                 OK: {"value":42}
   6. Structured Outputs        PARTIAL: 未遵循 schema（缺少必要字段）: ["c
   7. Function Calling          OK: {"city": "北京"}
   8. 并行工具调用              OK: 2个工具调用
   9. Vision 图片(URL)          OK: 这张图片是一个电商促销标识，内容为：\n\n- 上方是四个橙
  10. Base64 图片               OK: 这张图片是一个简洁的电商促销标识，背景为纯白色。\n\n主要
  11. temperature               OK: 参数已接受
  12. top_p                     OK: 参数已接受
  13. n 参数                    NOT SUPPORTED: API返回choices:null
  14. seed                      OK: 参数已接受
  15. logit_bias                OK: 参数已接受
  16. frequency_penalty         OK: 参数已接受
  17. presence_penalty          OK: 参数已接受
  18. stop 参数                 OK: finish=stop, content='小明'
  19. logprobs              